# LI-6800 control

Two ways to talk to the instrument:

- **Native MQTT (primary, recommended)** — `li_mqtt.LI6800` connects directly to the LI-6800's internal MQTT bus over link-local IPv6. Real-time data, direct setpoints, nothing to start on the console. This is the same bus the touchscreen itself uses.
- **SSH file-protocol (fallback)** — `li_connect.LiCor` drops JSON commands over SSH to the `RemoteEnvMeasure.py` background program. Works, but requires that BP to be started once on the console.

Run the install cell once, then use the MQTT section. **`li.set(...)` drives the chamber immediately** — treat it like turning a knob on the console.

In [ ]:
%pip install -r requirements.txt

## Native MQTT control

In [1]:
from li_mqtt import LI6800

# Discovers the head's MQTT broker on the wired link (validates live data flow)
# and subscribes to the measurement streams. No password, no background program.
li = LI6800.connect()
print("connected to broker", li.broker)

connected to broker fe80::21c:94ff:fe04:8d5c%6


In [5]:
# Live values: raw measured (CO2_r, H2O_s, Tleaf, ...) merged with computed
# gas exchange (A, gsw, Ci, E, ...). Real-time; reading changes nothing.
vals = li.read()
for k in ("CO2_r", "CO2_s", "H2O_r", "H2O_s", "Tleaf", "Tchamber",
          "RHcham", "VPDleaf", "Flow", "Fan_speed", "A", "gsw", "Ci", "E"):
    print(f"  {k:<10} {vals.get(k)}")
print(f"({len(vals)} values total — see li.read().keys())")

  CO2_r      47.4663
  CO2_s      3.48317
  H2O_r      19.7946
  H2O_s      19.8503
  Tleaf      25.2349
  Tchamber   25.8186
  RHcham     60.389738698425234
  VPDleaf    1.2085455787545256
  Flow       0.0259246
  Fan_speed  0
  A          0.07063849910296056
  gsw        0.0014057925795264148
  Ci         -76.29646769375765
  E          1.714882252139173e-05
(76 values total — see li.read().keys())


In [ ]:
# Set chamber conditions. THIS DRIVES THE HARDWARE IMMEDIATELY.
# Supported keys: co2_r, co2_s, co2_pct, flow, flow_pct, tair, tleaf, txchg,
#                 rh_air, vpd_leaf, sd_air, h2o_r, h2o_s, fan_rpm.
# Pass a value of None to turn a controller off.
li.set(co2_r=400, tair=25, rh_air=50)

In [ ]:
# Optionally block until a value settles, then read it back.
li.wait_stable("CO2_r", 400, tol=5, timeout_s=120)
print("CO2_r =", li.get("CO2_r"), "  Tleaf =", li.get("Tleaf"), "  A =", li.get("A"))

In [ ]:
li.close()   # disconnect from the bus when done (or use `with LI6800.connect() as li:`)

### MQTT reference

**Setpoints** (`li.set(**kwargs)`, value = `None` turns the controller off):

| Key | Sets | Units |
|---|---|---|
| `co2_r` / `co2_s` | Reference / sample CO₂ | µmol mol⁻¹ |
| `co2_pct` | CO₂ mixer % | % |
| `tair` / `tleaf` / `txchg` | Air / leaf / exchanger temperature | °C |
| `rh_air` / `vpd_leaf` / `sd_air` | Humidity by RH / VPD / SD | % / kPa / kPa |
| `h2o_r` / `h2o_s` | Reference / sample H₂O | mmol mol⁻¹ |
| `flow` / `flow_pct` | Chamber flow | µmol s⁻¹ / % |
| `fan_rpm` | Fan speed | rpm |

Light/`Qin` is not in the typed setters yet (the LI-6800 routes light across head / console / fluorometer sources); use `li.set_raw(topic, payload)` or a background program for now.

**Reading:** `li.read()` returns a dict; `li.get("A")` one value; `li.raw(topic)` a subscribed topic's last payload. Common keys: `CO2_r`, `CO2_s`, `H2O_r`, `H2O_s`, `Tleaf`, `Tchamber`, `RHcham`, `VPDleaf`, `Flow`, `Fan_speed`, `Press`, and computed `A`, `gsw`, `gbw`, `Ci`, `E`.

**Architecture:** the broker (Mosquitto, port 1883) runs on the sensor **head** (`li6850`); the console (`li6860`) and fluorometer (`li6840`) are clients. Topic strings can drift between Bluestem firmware versions — re-verify with `mqtt_sniff.py` if a future update breaks something.

**Background programs (experimental):** `li.run_bp("/home/licor/apps/.../X.py")`, `li.run_steps(steps)`, `li.stop_bp()`.

## Fallback: SSH + `RemoteEnvMeasure` background program

The original path: JSON commands shipped over SSH to a resident background program. Use it only if the MQTT route is unavailable. It **requires `RemoteEnvMeasure.py` to be running** on the console (start it from Bluestem ▸ Background Programs); `li_ssh.bp_running()` checks that. First connect installs an SSH key (factory password: `licor`); later runs are passwordless.

In [ ]:
from li_connect import LiCor

li_ssh = LiCor.connect()          # enter password 'licor' on first run; passwordless after
print("background program running:", li_ssh.bp_running())

In [ ]:
# Read current values (no setpoint change, no log record):
print(li_ssh.read())

# Set conditions and log one record (requires an open log file on the console):
ack = li_ssh.measure(co2_r=399, tair=25, rh_air=50, wait_for_co2=True, co2_tol=5, wait_s=1, log=True)
ack